# KazMMLU Kazakh STEM: evaluation

Scores one model on
* the five Kazakh STEM subjects of [KazMMLU](https://huggingface.co/datasets/MBZUAI/KazMMLU) (test split, about 4,200 questions),
* two Kazakh non-STEM subjects (Kazakh History, Geography), to check that fine-tuning doesn't damage other knowledge,
* the 413 held-out questions of Kazakh STEM MCQ 2000 (`test_own.csv`).

**Method.** Zero-shot. Each question is shown once with an English instruction; the model's probability for each answer letter is read off, and the most likely letter that exists in that question is its answer. Same code before and after fine-tuning: only `USE_ADAPTER` changes.

**How to run.** Right panel: Accelerator = GPU T4 x2, Internet = On, Add-ons → Secrets → HF_TOKEN switched on (needed for Llama). Add Input → Your Datasets → kazakh-stem-2000. Edit the settings cell, then Save Version → Save & Run All.

**Output:** `results_<RUN_NAME>.csv` (one row per question) and `summary_<RUN_NAME>.json`.


In [ ]:
# ===== Settings: change only these =====
MODEL = "llama"          # "llama" = meta-llama/Llama-3.1-8B-Instruct (needs approved Meta access)
                         # "qwen"  = Qwen/Qwen2.5-7B-Instruct (no access form)
USE_ADAPTER = False      # False = base model; True = base model + trained adapter
RUN_NAME = "base"        # names the output files, e.g. "base", "tuned_seed1"
BATCH_SIZE = 8           # lower to 4 if you see "CUDA out of memory"
INCLUDE_NONSTEM = True   # also score two non-STEM Kazakh subjects


In [ ]:
import glob, json, math, os, subprocess, sys, time
if USE_ADAPTER:      # Kaggle ships torchao 0.10, which peft refuses to load next to; this project does not use torchao
    import importlib.util
    if importlib.util.find_spec("torchao") is not None:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
        print("removed torchao (not used here; its old version blocks peft)")
import pandas as pd
import torch, transformers
from packaging import version

print("torch", torch.__version__, "| transformers", transformers.__version__)
n_gpu = torch.cuda.device_count()
mem = [torch.cuda.get_device_properties(i).total_memory / 2**30 for i in range(n_gpu)]
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(n_gpu)], [f"{m:.0f} GB" for m in mem])
if n_gpu == 0 or sum(mem) < 28:
    raise RuntimeError("Needs GPU T4 x2 (two 15 GB GPUs): right panel -> Session options -> Accelerator.")
if version.parse(transformers.__version__) < version.parse("4.45"):
    raise RuntimeError("transformers 4.45 or newer is required.")

HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN secret found")
except Exception as e:
    print("No HF_TOKEN secret (fine for Qwen, needed for Llama):", type(e).__name__)

if USE_ADAPTER:
    try:
        import peft
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft"], check=True)
        import peft
    print("peft", peft.__version__)


In [ ]:
# Shared helpers: the same prompt is used for evaluation and training.
import math

import pandas as pd

LETTERS = "ABCDE"
STEM_SUBSETS = ["Math (High School in kaz)", "Physics (High School in kaz)", "Chemistry (High School in kaz)",
                "Biology (High School in kaz)", "Informatics (High School in kaz)"]
NONSTEM_SUBSETS = ["Kazakh History (High School in kaz)", "Geography (High School in kaz)"]
SUBJECT_EN = {  # subset or own-subject -> words used in the prompt
    "Math (High School in kaz)": "mathematics", "Physics (High School in kaz)": "physics",
    "Chemistry (High School in kaz)": "chemistry", "Biology (High School in kaz)": "biology",
    "Informatics (High School in kaz)": "informatics", "Kazakh History (High School in kaz)": "history of Kazakhstan",
    "Geography (High School in kaz)": "geography",
    "math": "mathematics", "physics": "physics", "chemistry": "chemistry", "biology": "biology",
    "informatics": "informatics",
}


def build_messages(subject_en: str, question: str, options: dict) -> list:
    """The one prompt used everywhere: English instruction, Kazakh question and options."""
    letters = [L for L in LETTERS if str(options.get(L, "")).strip()]
    lines = [f"The following is a multiple choice question about {subject_en} (high school level, in Kazakh).",
             f"Reply with only the letter of the correct answer ({', '.join(letters)}).", "",
             str(question).strip(), ""]
    lines += [f"{L}. {str(options[L]).strip()}" for L in letters]
    lines += ["", "Answer:"]
    return [{"role": "user", "content": "\n".join(lines)}]


def items_from_kazmmlu(df: pd.DataFrame, subset: str, set_name: str):
    """Rows of a KazMMLU CSV -> list of item dicts. Read the CSV with dtype=str, keep_default_na=False
    so that options like 'NA' (sodium) or 'None' stay text."""
    items, skipped = [], 0
    for _, r in df.iterrows():
        opts = {L: str(r.get(f"Option {L}", "")).strip() for L in LETTERS}
        gold = str(r.get("Answer Key", "")).strip().upper()
        avail = [L for L in LETTERS if opts[L]]
        if gold not in avail or len(avail) < 2 or not str(r.get("Question", "")).strip():
            skipped += 1
            continue
        items.append(dict(set=set_name, subset=subset, item_id=str(r.get("ID", "")), gold=gold,
                          messages=build_messages(SUBJECT_EN[subset], r["Question"], opts), letters="".join(avail)))
    return items, skipped


def items_from_own(df: pd.DataFrame, set_name: str = "own_test"):
    """test_own.csv from validate_questions.py: id, subject, topic, question, A..E, answer."""
    items, skipped = [], 0
    for _, r in df.iterrows():
        opts = {L: str(r.get(L, "")).strip() for L in LETTERS}
        gold = str(r.get("answer", "")).strip().upper()
        avail = [L for L in LETTERS if opts[L]]
        if gold not in avail:
            skipped += 1
            continue
        items.append(dict(set=set_name, subset=str(r["subject"]), item_id=str(r["id"]), gold=gold,
                          messages=build_messages(SUBJECT_EN[str(r["subject"])], r["question"], opts),
                          letters="".join(avail)))
    return items, skipped


def pick_letter(letter_logprobs: dict, letters: str):
    """letter -> log-probability; only letters that exist in this question compete.
    Returns (prediction, probabilities renormalised over the available letters)."""
    avail = [L for L in letters]
    m = max(letter_logprobs[L] for L in avail)
    ex = {L: math.exp(letter_logprobs[L] - m) for L in avail}
    z = sum(ex.values())
    probs = {L: ex[L] / z for L in avail}
    return max(avail, key=lambda L: probs[L]), probs


def wilson(k: int, n: int, z: float = 1.96):
    if n == 0:
        return (float("nan"), float("nan"))
    p = k / n
    d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (c - h, c + h)


def summarize(res: pd.DataFrame) -> dict:
    """Accuracy per set and per subset with 95 % Wilson intervals, plus chance level."""
    out = {}
    for s, g in res.groupby("set"):
        k, n = int(g["correct"].sum()), len(g)
        per = {}
        for sub, h in g.groupby("subset"):
            kk, nn = int(h["correct"].sum()), len(h)
            per[sub] = dict(n=nn, accuracy=kk / nn, ci95=wilson(kk, nn), chance=float((1 / h["n_options"]).mean()))
        out[s] = dict(n=n, accuracy=k / n, ci95=wilson(k, n), chance=float((1 / g["n_options"]).mean()),
                      macro_accuracy=float(sum(v["accuracy"] for v in per.values()) / len(per)), by_subset=per)
    return out


In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

REPO = "MBZUAI/KazMMLU"
files = list_repo_files(REPO, repo_type="dataset", token=HF_TOKEN)

def test_file(subset):
    inside = [f for f in files if f.startswith(subset + "/") and f.lower().endswith(".csv")]
    tests = [f for f in inside if "test" in f.rsplit("/", 1)[-1].lower()]
    if len(tests) != 1:
        raise RuntimeError(f"{subset}: expected one test CSV, found {inside}")
    return tests[0]

items, skipped = [], {}
for sub in STEM_SUBSETS + (NONSTEM_SUBSETS if INCLUDE_NONSTEM else []):
    path = hf_hub_download(REPO, test_file(sub), repo_type="dataset", token=HF_TOKEN)
    df = pd.read_csv(path, dtype=str, keep_default_na=False)      # keep 'NA', 'None' as text
    got, sk = items_from_kazmmlu(df, sub, "kazmmlu_stem" if sub in STEM_SUBSETS else "kazmmlu_nonstem")
    items += got
    skipped[sub] = sk
    print(f"{sub:40s} {len(got):5d} questions ({sk} skipped: no valid answer key)")

own = sorted(glob.glob("/kaggle/input/**/test_own.csv", recursive=True))
if own:
    df = pd.read_csv(own[0], dtype=str, keep_default_na=False, encoding="utf-8-sig")
    got, sk = items_from_own(df)
    items += got
    skipped["own_test"] = sk
    print(f"{'held-out questions':40s} {len(got):5d} questions ({sk} skipped)  [{own[0]}]")
else:
    print("test_own.csv not found: Add Input -> Your Datasets -> kazakh-stem-2000. Continuing with KazMMLU only.")
print("total:", len(items))


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_IDS = {"llama": "meta-llama/Llama-3.1-8B-Instruct", "qwen": "Qwen/Qwen2.5-7B-Instruct"}
model_id = MODEL_IDS[MODEL]
try:
    tok = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
except Exception as e:
    if MODEL == "llama":
        raise RuntimeError("Can't open Llama: no access to the gated model, or the HF_TOKEN secret is off.") from e
    raise
tok.padding_side = "left"
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

t_load = time.time()
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN)
adapter_dir = None
if USE_ADAPTER:
    from peft import PeftModel
    cfgs = glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
    if len(cfgs) != 1:
        raise RuntimeError(f"Expected one adapter in the inputs (adapter_config.json), found {cfgs}. Add the training run's output as an input.")
    adapter_dir = os.path.dirname(cfgs[0])
    print("adapter:", adapter_dir, "| trained on:", json.load(open(cfgs[0])).get("base_model_name_or_path"))
    model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()
print(f"loaded {model_id} in {time.time() - t_load:.0f} s")


In [ ]:
def letter_ids(tok):
    ids = {}
    for L in LETTERS:
        c = set()
        for s in (L, " " + L):                       # 'A' and ' A' can be different tokens
            t = tok.encode(s, add_special_tokens=False)
            if len(t) == 1:
                c.add(t[0])
        if not c:
            raise RuntimeError(f"no single-token form of {L}")
        ids[L] = sorted(c)
    return ids

LIDS = letter_ids(tok)
print("letter token ids:", LIDS)
for it in items:                                     # the chat template is applied once per question
    it["text"] = tok.apply_chat_template(it["messages"], tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def score(batch):
    enc = tok([it["text"] for it in batch], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    lp = torch.log_softmax(model(**enc).logits[:, -1, :].float(), dim=-1)
    out = []
    for i, it in enumerate(batch):
        llp = {L: torch.logsumexp(lp[i, LIDS[L]], dim=0).item() for L in LETTERS}
        pred, probs = pick_letter(llp, it["letters"])
        out.append(dict(pred=pred, probs=probs, letter_mass=sum(math.exp(llp[L]) for L in it["letters"])))
    return out


In [ ]:
# Quick check on 2 questions per subject before the full run
probe = []
for sub in dict.fromkeys(it["subset"] for it in items):
    probe += [it for it in items if it["subset"] == sub][:2]
print("----- what the model sees (first question) -----")
print(probe[0]["text"])
print("------------------------------------------------")
probe_res = score(probe)
for it, r in zip(probe, probe_res):
    print(f"{it['subset'][:32]:32s} gold {it['gold']}  pred {r['pred']}  p={r['probs'][r['pred']]:.2f}  "
          f"probability on answer letters {r['letter_mass']:.2f}")
mass = sum(r["letter_mass"] for r in probe_res) / len(probe)
print(f"mean probability on answer letters: {mass:.2f}")
if mass < 0.5:
    print("WARNING: the model puts little probability on the answer letters. Check the prompt before trusting the scores.")


In [ ]:
order = sorted(range(len(items)), key=lambda i: len(items[i]["text"]))   # similar lengths together = less padding
rows = [None] * len(items)
t0 = time.time()
for b in range(0, len(order), BATCH_SIZE):
    idx = order[b:b + BATCH_SIZE]
    for i, r in zip(idx, score([items[i] for i in idx])):
        it = items[i]
        rows[i] = dict(set=it["set"], subset=it["subset"], item_id=it["item_id"], gold=it["gold"], pred=r["pred"],
                       correct=int(r["pred"] == it["gold"]), n_options=len(it["letters"]),
                       letter_mass=round(r["letter_mass"], 4),
                       **{f"p_{L}": round(r["probs"].get(L, 0.0), 4) for L in LETTERS})
    if (b // BATCH_SIZE) % 50 == 0:
        done, el = b + len(idx), time.time() - t0
        print(f"{done}/{len(items)} questions, {el / 60:.1f} min, about {el / done * (len(items) - done) / 60:.0f} min left", flush=True)
res = pd.DataFrame(rows)
res.to_csv(f"/kaggle/working/results_{RUN_NAME}.csv", index=False)
print(f"done in {(time.time() - t0) / 60:.1f} min")


In [ ]:
summary = dict(run=RUN_NAME, model=model_id, adapter=adapter_dir,
               method="zero-shot, English instruction, chat template, answer-letter log-probability",
               dtype="float16", transformers=transformers.__version__, torch=torch.__version__,
               minutes=round((time.time() - t0) / 60, 1), skipped=skipped,
               mean_letter_mass=round(float(res["letter_mass"].mean()), 4), results=summarize(res))
json.dump(summary, open(f"/kaggle/working/summary_{RUN_NAME}.json", "w"), indent=1, ensure_ascii=False)
for s, v in summary["results"].items():
    lo, hi = v["ci95"]
    print(f"\n{s}: {v['accuracy']:.1%} (95% CI {lo:.1%} to {hi:.1%}), n = {v['n']}, guessing would give {v['chance']:.0%}")
    for sub, w in v["by_subset"].items():
        print(f"    {sub:40s} {w['accuracy']:.1%}   n = {w['n']}")
print(f"\nmean probability on answer letters: {summary['mean_letter_mass']:.2f}")
print(f"saved results_{RUN_NAME}.csv and summary_{RUN_NAME}.json")
